# Hybrid Search RAG — LangChain + BM25 + Sentence Transformers + Gemini

**Goal:** compare lexical BM25 retrieval, semantic dense retrieval, and **hybrid retrieval** (weighted Reciprocal Rank Fusion) on a small in-memory list. Then send the retrieved context to the Google Gemini API to answer the query.

**Run in Google Colab from top to bottom.** The embedding model runs locally; a Google API key is needed only for the final generation step. Sample documents are deliberately written in **English** because the selected embedding model is English-focused.

> Hybrid search is *retrieval fusion*, not contextual compression. We do **not** pass BM25 scores and cosine scores directly into the same sum.

## Install dependencies

In [ ]:
!pip -q install -U langchain-core langchain-community langchain-classic langchain-huggingface langchain-google-genai sentence-transformers rank-bm25

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## Sample data: a Python list of dictionaries

In [ ]:
from langchain_core.documents import Document

raw_data = [
    {"id":"D01", "title":"Dense embeddings", "text":"Sentence Transformers transform sentences into dense vectors. Semantic search compares their vector representations to retrieve passages with similar meanings even when exact terms differ."},
    {"id":"D02", "title":"BM25 lexical search", "text":"BM25 is a keyword retrieval algorithm. It scores documents using term frequency, inverse document frequency, and document length normalization. It matches exact query terms."},
    {"id":"D03", "title":"Hybrid search", "text":"Hybrid search combines BM25 keyword retrieval with dense semantic vector search. Reciprocal Rank Fusion (RRF) merges ranked results from two retrievers without requiring comparable raw scores."},
    {"id":"D04", "title":"Neural reranking", "text":"A cross-encoder reranker scores each query-document pair jointly. Reranking improves ordering of a candidate set but is more expensive than initial retrieval."},
    {"id":"D05", "title":"Text chunking", "text":"Chunking splits long source documents into smaller passages before embedding and indexing. Chunk size and overlap affect retrieval granularity."},
    {"id":"D06", "title":"Vector similarity", "text":"Cosine similarity measures the angle between vector embeddings. Nearest-neighbor search retrieves geometrically similar vectors in embedding space."},
    {"id":"D07", "title":"Contextual compression", "text":"Contextual compression removes irrelevant text from retrieved documents based on the user query. It is different from hybrid search, which combines retrieval strategies."},
    {"id":"D08", "title":"RAG generation", "text":"Retrieval-augmented generation supplies retrieved passages as context to a language model. The model should answer from the provided context and acknowledge missing evidence."},
    {"id":"D09", "title":"Python dictionaries", "text":"Python dictionaries map keys to values. Lists hold ordered sequences of data. Both are common structures for storing sample datasets."},
    {"id":"D10", "title":"Search synonyms", "text":"Automobiles and cars have similar meanings. Semantic retrieval may connect a question about an automobile to a passage discussing a car even without matching identical words."},
]

documents = [Document(page_content=row['text'], metadata={'id':row['id'], 'title':row['title']}) for row in raw_data]
print(f'{len(documents)} documents loaded')
print(documents[0])

10 documents loaded
page_content='Sentence Transformers transform sentences into dense vectors. Semantic search compares their vector representations to retrieve passages with similar meanings even when exact terms differ.' metadata={'id': 'D01', 'title': 'Dense embeddings'}


## Sparse / keyword retrieval: BM25

In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(
    documents,
    preprocess_func=lambda text: text.lower().split(),
)
bm25_retriever.k = 4
print('BM25 ready')

BM25 ready


## Dense retrieval: Sentence Transformers + in-memory vector store

`HuggingFaceEmbeddings` uses the Sentence Transformers model locally; **no Google Embeddings API is used**.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(documents)
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 4})
print('Dense retriever ready')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dense retriever ready


## Hybrid retrieval: weighted Reciprocal Rank Fusion (RRF)

`EnsembleRetriever` combines **rank positions**, not raw BM25 and cosine similarity scores. `weights=[0.5, 0.5]` gives each ranking equal weight. `id_key="id"` deduplicates documents by metadata.

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
    c=60,
    id_key="id",
)
print('Hybrid retriever ready')

Hybrid retriever ready


## Compare retrieval outputs

In [ ]:
query = "How can I combine exact keyword matching and semantic similarity in a RAG search system?"

def show_results(label, found):
    print(f"\n--- {label} ({len(found)} results) ---")
    for rank, doc in enumerate(found, 1):
        print(f"{rank}. [{doc.metadata['id']}] {doc.metadata['title']} — {doc.page_content[:110]}...")

bm25_docs = bm25_retriever.invoke(query)
dense_docs = dense_retriever.invoke(query)
hybrid_docs = hybrid_retriever.invoke(query)
show_results('BM25', bm25_docs)
show_results('Dense', dense_docs)
show_results('Hybrid (RRF)', hybrid_docs)


--- BM25 (4 results) ---
1. [D06] Vector similarity — Cosine similarity measures the angle between vector embeddings. Nearest-neighbor search retrieves geometricall...
2. [D10] Search synonyms — Automobiles and cars have similar meanings. Semantic retrieval may connect a question about an automobile to a...
3. [D02] BM25 lexical search — BM25 is a keyword retrieval algorithm. It scores documents using term frequency, inverse document frequency, a...
4. [D01] Dense embeddings — Sentence Transformers transform sentences into dense vectors. Semantic search compares their vector representa...

--- Dense (4 results) ---
1. [D03] Hybrid search — Hybrid search combines BM25 keyword retrieval with dense semantic vector search. Reciprocal Rank Fusion (RRF) ...
2. [D01] Dense embeddings — Sentence Transformers transform sentences into dense vectors. Semantic search compares their vector representa...
3. [D10] Search synonyms — Automobiles and cars have similar meanings. Semantic retrieval may c

## Optional: change the query and rerun retrieval

In [ ]:
new_query = "Why use RRF to mix lexical BM25 with vector retrieval?"
show_results('Hybrid new query', hybrid_retriever.invoke(new_query))


--- Hybrid new query (5 results) ---
1. [D03] Hybrid search — Hybrid search combines BM25 keyword retrieval with dense semantic vector search. Reciprocal Rank Fusion (RRF) ...
2. [D01] Dense embeddings — Sentence Transformers transform sentences into dense vectors. Semantic search compares their vector representa...
3. [D02] BM25 lexical search — BM25 is a keyword retrieval algorithm. It scores documents using term frequency, inverse document frequency, a...
4. [D06] Vector similarity — Cosine similarity measures the angle between vector embeddings. Nearest-neighbor search retrieves geometricall...
5. [D08] RAG generation — Retrieval-augmented generation supplies retrieved passages as context to a language model. The model should an...


## Use Gemini only for answer generation

Change the model name if your Google API project does not have access to the default. The prompt says to use retrieved context only.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's question using only the provided context. If the context is insufficient, say so. Cite document IDs like [D03]."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])
chain = prompt | llm | StrOutputParser()

context = "\n\n".join(
    f"[{doc.metadata['id']}] {doc.metadata['title']}\n{doc.page_content}"
    for doc in hybrid_docs[:4]
)
answer = chain.invoke({"context": context, "question": query})
print('QUESTION:', query)
print('\nANSWER:\n', answer)

QUESTION: How can I combine exact keyword matching and semantic similarity in a RAG search system?

ANSWER:
 Based on the provided context, there is insufficient information to answer how to combine exact keyword matching and semantic similarity in a RAG search system. The context describes BM25 for keyword retrieval [D02] and vector embeddings for semantic search [D01, D06], but it does not explain how to combine them.
